# Part 3: Key RL Algorithms — DP, Monte Carlo, Q-Learning & SARSA

**Course**: Reinforcement Learning for AI/ML Engineers  
**Duration**: 1.5 Hours  
**Instructor**: Mehdi Lotfinejad  

---

## Learning Objectives

By the end of this section, you will be able to:

1. **Implement** Dynamic Programming (Policy Iteration & Value Iteration) for solving MDPs with a known model
2. **Apply** Monte Carlo methods to learn from complete episodes — no environment model needed
3. **Build** Q-Learning from scratch — the famous off-policy TD algorithm that powers real-world RL
4. **Implement** SARSA — the on-policy alternative that behaves more safely than Q-Learning
5. **Compare** all four algorithms side-by-side and know when to use each one
6. **Train** agents on Gymnasium environments: FrozenLake, CliffWalking, and Taxi

---

## 1. Why Do We Need Multiple Algorithms?

In Parts 1 and 2, we already saw two algorithms in action:
- **Q-Learning** (Part 1) — taught a grid-world agent by trial and error
- **Value Iteration** (Part 2) — found the optimal strategy by solving the Bellman equation exactly

But a major question was left unanswered: **which algorithm should I use for my problem?**

The answer depends on a single key question:

> **Do you know the rules of the game?**

| Situation | Algorithm Family | Example |
|---|---|---|
| **You have the full model** (you know all transitions and rewards) | Dynamic Programming | Solving a known MDP like the Student MDP from Part 2 |
| **No model — you learn from complete episodes** | Monte Carlo | Learning Blackjack by playing lots of hands |
| **No model — you learn step by step** | Temporal Difference (Q-Learning, SARSA) | Teaching a robot via live interaction |

### The Algorithm Landscape

```
                         ALL RL ALGORITHMS
                                |
             ┌──────────────────┼──────────────────┐
             |                  |                  |
       Model-Based          Model-Free         Model-Based
     (Know the rules)     (Learn by doing)    + Model-Free
             |                  |
    Dynamic Programming    ┌────┴────────────────┐
    (THIS PART: Section 2) |                     |
                     Monte Carlo           Temporal Difference
                   (Section 3)           (Section 4 & 5)
                    • MC Prediction         • Q-Learning (off-policy)
                    • MC Control            • SARSA (on-policy)
                                            • TD(0)
```

### One Unifying Goal

Despite their differences, every algorithm in this section is trying to answer the **same question**:

> *What is the best action to take in each state?*

They just take different routes to get there. Let's explore each one!

## 2. Dynamic Programming — Solving MDPs with a Perfect Map

**Dynamic Programming (DP)** assumes you have a **complete model** of the environment — you know exactly what happens when you take every action in every state.

Think of it like using Google Maps with perfect traffic data: you know every road, every turn, the exact travel time. You can calculate the best route **without driving it first**.

### The Two Core DP Algorithms

| Algorithm | What It Does | When to Use |
|---|---|---|
| **Policy Iteration** | Alternates between evaluating a policy and improving it | When you want to track which policy you have at each step |
| **Value Iteration** | Directly computes optimal values (collapses eval + improvement into one step) | Faster in practice — usually preferred |

We already used Value Iteration in Part 2. Let's now build **Policy Iteration** and understand the difference.

### 2.1 Policy Iteration — Evaluate, Improve, Repeat

Policy Iteration cycles between two steps:

```
Step 1 — Policy Evaluation:
  Given a fixed policy π, compute how good each state is: V^π(s)
  (Solve Bellman equation for the current policy)

Step 2 — Policy Improvement:
  For each state, check: "Is there a better action than what my policy says?"
  If yes → update the policy to take that better action

Repeat until the policy stops changing → YOU HAVE AN OPTIMAL POLICY!
```

Mathematically, the improvement step is:

$$\pi'(s) = \arg\max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma \cdot V^\pi(s') \right]$$

In plain English: *"Look at your current state values, and switch to whichever action leads to the best combination of immediate reward + future value."*

In [12]:
import numpy as np

# ============================================================
# Policy Iteration on the Student MDP (from Part 2)
# ============================================================

# Re-define the Student MDP (same as Part 2)
class StudentMDP:
    def __init__(self):
        self.states = ['Class1', 'Class2', 'Class3', 'Facebook', 'Sleep', 'Pass']
        self.terminal_states = {'Sleep', 'Pass'}
        self.actions = {
            'Class1':   ['Study', 'Facebook'],
            'Class2':   ['Study', 'Sleep'],
            'Class3':   ['Study', 'Pub'],
            'Facebook': ['Facebook', 'Quit'],
            'Sleep': [], 'Pass': [],
        }
        self.transitions = {
            ('Class1', 'Study'):      [(1.0, 'Class2', -2)],
            ('Class1', 'Facebook'):   [(1.0, 'Facebook', -1)],
            ('Class2', 'Study'):      [(1.0, 'Class3', -2)],
            ('Class2', 'Sleep'):      [(1.0, 'Sleep', 0)],
            ('Class3', 'Study'):      [(1.0, 'Pass', +10)],
            ('Class3', 'Pub'):        [(0.4, 'Class1', +1), (0.4, 'Class2', +1), (0.2, 'Class3', +1)],
            ('Facebook', 'Facebook'): [(1.0, 'Facebook', -1)],
            ('Facebook', 'Quit'):     [(1.0, 'Class1', 0)],
        }
    def get_transitions(self, s, a): return self.transitions.get((s, a), [])
    def get_actions(self, s):        return self.actions.get(s, [])
    def is_terminal(self, s):        return s in self.terminal_states

def policy_evaluation(mdp, policy, gamma=0.9, threshold=1e-8):
    """Evaluate a policy: compute V^pi(s) for all states."""
    V = {s: 0.0 for s in mdp.states}
    for iteration in range(10000):
        max_change = 0
        for state in mdp.states:
            if mdp.is_terminal(state):
                continue
            action = policy.get(state)
            if action is None:
                continue
            new_v = sum(p * (r + gamma * V[ns])
                        for p, ns, r in mdp.get_transitions(state, action))
            max_change = max(max_change, abs(new_v - V[state]))
            V[state] = new_v
        if max_change < threshold:
            break
    return V

def policy_improvement(mdp, V, gamma=0.9):
    """One round of greedy policy improvement."""
    new_policy = {}
    policy_stable = True
    for state in mdp.states:
        if mdp.is_terminal(state):
            continue
        actions = mdp.get_actions(state)
        if not actions:
            continue
        old_action = new_policy.get(state)
        # Greedy: pick the action with highest Q-value
        q_values = {
            a: sum(p * (r + gamma * V[ns]) for p, ns, r in mdp.get_transitions(state, a))
            for a in actions
        }
        best_action = max(q_values, key=q_values.get)
        new_policy[state] = best_action
        if old_action is not None and old_action != best_action:
            policy_stable = False
    return new_policy, policy_stable

def policy_iteration(mdp, gamma=0.9):
    """Full Policy Iteration: alternate evaluation and improvement."""
    # Start with an arbitrary policy (always pick first available action)
    policy = {s: mdp.get_actions(s)[0] for s in mdp.states if mdp.get_actions(s)}
    
    print("=" * 60)
    print("POLICY ITERATION — Step by Step")
    print("=" * 60)
    
    for iteration in range(100):
        print(f"\n--- Iteration {iteration + 1} ---")
        print(f"  Current policy: {policy}")
        
        # Step 1: Evaluate the current policy
        V = policy_evaluation(mdp, policy, gamma)
        print(f"  Values after evaluation:")
        for s in mdp.states:
            print(f"    V({s}) = {V[s]:+.4f}")
        
        # Step 2: Improve the policy
        policy, stable = policy_improvement(mdp, V, gamma)
        
        if stable:
            print(f"\n  Policy is STABLE — we have found the optimal policy!")
            break
    
    return V, policy

mdp = StudentMDP()
V_pi, optimal_policy = policy_iteration(mdp)

print("\n" + "=" * 60)
print("OPTIMAL POLICY FOUND:")
print("=" * 60)
for state, action in optimal_policy.items():
    print(f"  In {state:<12} → always '{action}'")
print("\nConclusion: Policy Iteration confirms — ALWAYS STUDY!")

POLICY ITERATION — Step by Step

--- Iteration 1 ---
  Current policy: {'Class1': 'Study', 'Class2': 'Study', 'Class3': 'Study', 'Facebook': 'Facebook'}
  Values after evaluation:
    V(Class1) = +4.3000
    V(Class2) = +7.0000
    V(Class3) = +10.0000
    V(Facebook) = -10.0000
    V(Sleep) = +0.0000
    V(Pass) = +0.0000

  Policy is STABLE — we have found the optimal policy!

OPTIMAL POLICY FOUND:
  In Class1       → always 'Study'
  In Class2       → always 'Study'
  In Class3       → always 'Study'
  In Facebook     → always 'Quit'

Conclusion: Policy Iteration confirms — ALWAYS STUDY!


### 2.2 DP on FrozenLake — A Real Gymnasium Environment

Now let's use DP on a real Gymnasium environment: **FrozenLake**.

```
┌───┬───┬───┬───┐
│ S │   │   │   │    S = Start
├───┼───┼───┼───┤    F = Frozen (safe)
│   │ H │   │ H │    H = Hole  (fall in → -1, episode ends)
├───┼───┼───┼───┤    G = Goal  (+1, episode ends)
│   │   │   │ H │
├───┼───┼───┼───┤    Stochastic: the ice is slippery!
│ H │   │   │ G │    Intended action succeeds only 1/3 of the time.
└───┴───┴───┴───┘
```

Gymnasium gives us access to `env.P` — the full transition matrix. That's all DP needs!

In [2]:
import gymnasium as gym
import numpy as np

# ============================================================
# Policy Iteration on FrozenLake-v1
# ============================================================

env = gym.make('FrozenLake-v1', is_slippery=True)
nS = env.observation_space.n   # 16 states (4x4 grid)
nA = env.action_space.n        # 4 actions: 0=Left, 1=Down, 2=Right, 3=Up
P  = env.unwrapped.P           # Full transition model: P[s][a] = [(prob, next_s, reward, done)]

action_symbols = {0: chr(8592), 1: chr(8595), 2: chr(8594), 3: chr(8593)}  # ← ↓ → ↑

def dp_policy_evaluation(P, policy, nS, gamma=0.99, threshold=1e-8):
    V = np.zeros(nS)
    while True:
        delta = 0
        for s in range(nS):
            a = policy[s]
            v = sum(prob * (r + gamma * V[ns] * (not done))
                    for prob, ns, r, done in P[s][a])
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < threshold:
            break
    return V

def dp_policy_improvement(P, V, nS, nA, gamma=0.99):
    policy = np.zeros(nS, dtype=int)
    for s in range(nS):
        q = np.array([
            sum(prob * (r + gamma * V[ns] * (not done)) for prob, ns, r, done in P[s][a])
            for a in range(nA)
        ])
        policy[s] = np.argmax(q)
    return policy

def dp_policy_iteration(P, nS, nA, gamma=0.99):
    policy = np.zeros(nS, dtype=int)   # Start: always go Left
    for iteration in range(1000):
        V = dp_policy_evaluation(P, policy, nS, gamma)
        new_policy = dp_policy_improvement(P, V, nS, nA, gamma)
        if np.array_equal(new_policy, policy):
            print(f"  Policy Iteration converged in {iteration + 1} iterations.")
            return V, new_policy
        policy = new_policy
    return V, policy

print("=" * 55)
print("POLICY ITERATION on FrozenLake-v1")
print("=" * 55)

V_frozen, opt_policy_frozen = dp_policy_iteration(P, nS, nA)

print("\nOptimal Values V*(s):")
print(V_frozen.reshape(4, 4).round(3))

print("\nOptimal Policy (arrows show best action per cell):")
grid = np.array([action_symbols[a] for a in opt_policy_frozen]).reshape(4, 4)
holes = [5, 7, 11, 12]   # H positions in FrozenLake
for r in range(4):
    row = ""
    for c in range(4):
        idx = r * 4 + c
        if idx == 15:            row += " G "
        elif idx in holes:       row += " H "
        else:                    row += f" {grid[r, c]} "
    print(row)

env.close()

# Quick evaluation
env2 = gym.make('FrozenLake-v1', is_slippery=True)
wins = 0
for _ in range(1000):
    s, _ = env2.reset()
    done = False
    while not done:
        s, r, terminated, truncated, _ = env2.step(int(opt_policy_frozen[s]))
        done = terminated or truncated
    if r == 1.0:
        wins += 1
env2.close()
print(f"\nDP Policy win rate (1,000 episodes): {wins / 10:.1f}%")

POLICY ITERATION on FrozenLake-v1
  Policy Iteration converged in 7 iterations.

Optimal Values V*(s):
[[0.542 0.499 0.471 0.457]
 [0.558 0.    0.358 0.   ]
 [0.592 0.643 0.615 0.   ]
 [0.    0.742 0.863 0.   ]]

Optimal Policy (arrows show best action per cell):
 ←  ↑  ↑  ↑ 
 ←  H  ←  H 
 ↑  ↓  ←  H 
 H  →  ↓  G 

DP Policy win rate (1,000 episodes): 74.6%


### 2.3 The Limits of Dynamic Programming

DP guarantees the **optimal solution** — so why don't we always use it?

| DP Advantage ✅ | DP Limitation ⚠️ |
|---|---|
| Finds the exact optimal policy | **Needs the full model** — transition probabilities P(s'\|s,a) |
| Provably correct | **Scales poorly** — for large state spaces (e.g., Atari: 10^70 states!), the tables are enormous |
| Deterministic and predictable | Real environments often **don't expose the model** |

> **Bottom line**: DP is perfect for small, fully-known environments. For anything larger or when you can't inspect the environment's internal rules, you need **model-free** methods.

That's where **Monte Carlo** comes in!

## 3. Monte Carlo Methods — Learning from Complete Episodes

**Monte Carlo (MC)** methods answer the question: *Can we learn good policies without knowing the model?*

The answer is **YES** — by playing complete episodes and learning from the outcomes.

### The Big Idea

> Think of learning to play poker: you don't know the mathematical probabilities of every hand combination. But if you play 10,000 hands, you start to *know* — from experience — which starting hands are worth playing. That's Monte Carlo.

### 3.1 MC Prediction — Estimating State Values

**Goal**: Estimate V(s) — how much total reward can I expect from each state?

**The method**:
1. Play a complete episode until the end
2. Look at every state you visited
3. For each visited state, compute the total discounted reward from that point onward (called the **return** G)
4. After many episodes, the average return per state becomes a good estimate of V(s)

```
Episode:   S0 → S1 → S2 → S3 (terminal)
Rewards:   r1    r2    r3

Return from S0:  G0 = r1 + γ·r2 + γ²·r3
Return from S1:  G1 = r2 + γ·r3
Return from S2:  G2 = r3

After many episodes: V(s) ≈ average of all G values observed from s
```

### 3.2 First-Visit vs. Every-Visit MC

| Variant | What gets averaged |
|---|---|
| **First-Visit MC** | Only the return from the **first time** you visit state s in an episode |
| **Every-Visit MC** | The return from **every time** you visit state s (even if you visit it multiple times in one episode) |

Both converge to the true value function — First-Visit is easier to implement and most commonly used.

In [3]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

# ============================================================
# Monte Carlo Prediction on Blackjack
# Blackjack is perfect for MC: no known transition model,
# but episodes are finite and clearly defined.
# ============================================================

env = gym.make('Blackjack-v1')

# A random policy (hit=1 if sum < 18, else stick=0)
def simple_policy(obs):
    player_sum, dealer_card, usable_ace = obs
    return 1 if player_sum < 18 else 0   # 1 = hit, 0 = stick

def mc_prediction_first_visit(env, policy_fn, n_episodes=100000, gamma=1.0):
    """First-Visit MC Prediction: estimate V(s) by averaging returns."""
    returns_sum   = defaultdict(float)
    returns_count = defaultdict(int)
    
    for ep in range(n_episodes):
        # Generate one complete episode
        episode = []
        obs, _ = env.reset()
        done = False
        while not done:
            action = policy_fn(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            episode.append((obs, action, reward))
            done = terminated or truncated
            obs = next_obs
        
        # Calculate returns backwards through the episode
        G = 0
        visited_states = set()
        for obs_t, _, reward in reversed(episode):
            G = reward + gamma * G
            state_key = obs_t  # use the observation as the state key
            if state_key not in visited_states:  # First-visit check
                visited_states.add(state_key)
                returns_sum[state_key]   += G
                returns_count[state_key] += 1
    
    V = {s: returns_sum[s] / returns_count[s] for s in returns_sum}
    return V

print("=" * 55)
print("MONTE CARLO PREDICTION on Blackjack")
print("=" * 55)
print("Running 100,000 episodes (this may take a moment)...")

V_blackjack = mc_prediction_first_visit(env, simple_policy, n_episodes=100_000)

# Show some representative values (player_sum, dealer_card, no ace)
print("\nEstimated V(s) — No Usable Ace, Dealer shows 6 (dealer is weak):")
print(f"  {'Player Sum':<14} {'V(s)':>8}")
print(f"  {'─'*24}")
for player_sum in range(12, 22):
    key = (player_sum, 6, False)
    v = V_blackjack.get(key, float('nan'))
    bar = chr(9608) * int(max(0, (v + 1) * 10))
    print(f"  Sum = {player_sum:<5}     V = {v:>+.3f}  {bar}")

print("\nHigher V → better chance of winning from that state!")
print("Notice: high sums (18+) with a weak dealer are favourable.")
env.close()

MONTE CARLO PREDICTION on Blackjack
Running 100,000 episodes (this may take a moment)...

Estimated V(s) — No Usable Ace, Dealer shows 6 (dealer is weak):
  Player Sum         V(s)
  ────────────────────────
  Sum = 12        V = -0.261  ███████
  Sum = 13        V = -0.355  ██████
  Sum = 14        V = -0.439  █████
  Sum = 15        V = -0.454  █████
  Sum = 16        V = -0.472  █████
  Sum = 17        V = -0.499  █████
  Sum = 18        V = +0.267  ████████████
  Sum = 19        V = +0.507  ███████████████
  Sum = 20        V = +0.713  █████████████████
  Sum = 21        V = +0.907  ███████████████████

Higher V → better chance of winning from that state!
Notice: high sums (18+) with a weak dealer are favourable.


### 3.3 MC Control — Finding the Optimal Policy

MC Prediction tells us the value of states. But to **act optimally**, we need Q-values: Q(s, a) — *how good is action a in state s?*

**MC Control** extends MC Prediction by:
1. Estimating **Q(s, a)** instead of V(s)
2. Improving the policy greedily after each episode (like Policy Improvement in DP)
3. Using **ε-greedy** exploration to ensure every state-action pair gets visited

```
MC Control loop:

Repeat many times:
  1. Generate episode using ε-greedy policy
  2. For each (s, a) visited:
       a. Compute return G from that (s, a) pair
       b. Update Q(s, a) ← average of all observed returns
  3. Improve policy: π(s) = argmax_a Q(s, a)  (greedy)
  4. Decay ε so exploration decreases over time
```

This guarantees convergence to the optimal policy (under GLIE conditions — Greedy in the Limit with Infinite Exploration).

In [4]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

# ============================================================
# MC Control (ε-greedy) on Blackjack
# ============================================================

env = gym.make('Blackjack-v1')

def mc_control(env, n_episodes=500_000, gamma=1.0, epsilon_start=1.0, epsilon_decay=0.999995):
    """On-policy MC Control with epsilon-greedy exploration."""
    Q        = defaultdict(lambda: np.zeros(env.action_space.n))
    returns  = defaultdict(list)
    epsilon  = epsilon_start
    wins = 0

    win_rates = []

    for ep in range(1, n_episodes + 1):
        episode = []
        obs, _ = env.reset()
        done = False
        while not done:
            # ε-greedy action selection
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[obs]))
            next_obs, reward, terminated, truncated, _ = env.step(action)
            episode.append((obs, action, reward))
            done = terminated or truncated
            obs = next_obs
        
        # Track wins
        if episode[-1][2] == 1.0:
            wins += 1

        # First-visit MC update for Q(s, a)
        G = 0
        visited = set()
        for s_t, a_t, r_t in reversed(episode):
            G = r_t + gamma * G
            if (s_t, a_t) not in visited:
                visited.add((s_t, a_t))
                returns[(s_t, a_t)].append(G)
                Q[s_t][a_t] = np.mean(returns[(s_t, a_t)])
        
        # Decay exploration
        epsilon = max(0.01, epsilon * epsilon_decay)

        if ep % 50000 == 0:
            win_rate = wins / 50000 * 100
            win_rates.append(win_rate)
            print(f"  Episodes {ep//1000:>4}k  |  Win rate: {win_rate:.1f}%  |  ε = {epsilon:.4f}")
            wins = 0

    return Q, win_rates

print("=" * 55)
print("MC CONTROL on Blackjack (500k episodes)")
print("=" * 55)
print(f"  {'Episode':>10}  {'Win Rate':>10}  {'Epsilon':>10}")
print("  " + "-" * 35)

Q_bj, win_rates = mc_control(env, n_episodes=500_000)

# Show learned policy for no usable ace
print("\nLearned Policy — No Usable Ace:")
print("  Player Sum   |  Dealer Card →  A  2  3  4  5  6  7  8  9  10")
print("  " + "-" * 55)
for player_sum in range(12, 22):
    row = f"  Sum = {player_sum:<5}  |  "
    for dealer in range(1, 11):
        key = (player_sum, dealer, False)
        if key in Q_bj:
            best = np.argmax(Q_bj[key])
            row += " H" if best == 1 else " S"
        else:
            row += " ?"
    print(row)

print("\n  H = Hit  |  S = Stick")
print("\nMC control is learning a reasonable Blackjack strategy!")
env.close()

MC CONTROL on Blackjack (500k episodes)
     Episode    Win Rate     Epsilon
  -----------------------------------
  Episodes   50k  |  Win rate: 30.1%  |  ε = 0.7788
  Episodes  100k  |  Win rate: 33.2%  |  ε = 0.6065
  Episodes  150k  |  Win rate: 35.6%  |  ε = 0.4724
  Episodes  200k  |  Win rate: 37.1%  |  ε = 0.3679
  Episodes  250k  |  Win rate: 38.7%  |  ε = 0.2865
  Episodes  300k  |  Win rate: 39.2%  |  ε = 0.2231
  Episodes  350k  |  Win rate: 40.0%  |  ε = 0.1738
  Episodes  400k  |  Win rate: 41.2%  |  ε = 0.1353
  Episodes  450k  |  Win rate: 41.4%  |  ε = 0.1054
  Episodes  500k  |  Win rate: 42.1%  |  ε = 0.0821

Learned Policy — No Usable Ace:
  Player Sum   |  Dealer Card →  A  2  3  4  5  6  7  8  9  10
  -------------------------------------------------------
  Sum = 12     |   H S S S S S H H H H
  Sum = 13     |   H S S S S S H H H S
  Sum = 14     |   H S S S S S H H H S
  Sum = 15     |   H S S S S S S S S S
  Sum = 16     |   H S S S S S S S S S
  Sum = 17     |

### 3.4 MC vs. DP — Key Differences

| | Dynamic Programming | Monte Carlo |
|---|---|---|
| **Model needed?** | ✅ Yes — needs P(s'\|s,a) | ❌ No — only needs experience |
| **Full episodes needed?** | No — can update at any step | ✅ Yes — wait until episode ends |
| **Works on continuing tasks?** | ✅ Yes | ❌ Needs terminal states |
| **Bootstrap?** | ✅ Uses other value estimates | ❌ Uses actual returns (no bootstrap) |
| **Guaranteed optimal?** | ✅ Yes (exact solution) | Under GLIE conditions |

**The missing ingredient**: What if we don't have the model AND need to update step-by-step (not wait for episode end)? That's what **Temporal Difference** methods solve!

## 4. Q-Learning — Off-Policy Temporal Difference Control

**Temporal Difference (TD)** learning is the best of both worlds — it learns **without a model** (like MC) AND **updates at every single step** (like DP does).

The key idea? Instead of waiting for the full return, TD **bootstraps** — it estimates the return using its current guess of the next state's value:

```
Monte Carlo:   V(s_t) ← V(s_t) + α [ G_t               − V(s_t) ]
                                      ↑                   ↑
                                   Actual return       Current estimate
                                   (wait for end)

TD(0):         V(s_t) ← V(s_t) + α [ r_{t+1} + γ·V(s_{t+1}) − V(s_t) ]
                                      ↑                              ↑
                                   TD Target               Current estimate
                                   (one step ahead!)
```

### 4.1 Q-Learning — The Famous Algorithm

Q-Learning extends TD(0) to learn **action values Q(s, a)** instead of state values. It is **off-policy**: the agent learns the optimal Q-values regardless of how it's currently exploring.

**The Q-Learning update rule**:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \right]$$

In plain English:
> *"Update my estimate of Q(s,a) toward: the actual reward I got + the highest Q-value possible from the next state."*

The crucial part is **max** — Q-Learning always targets the best possible next action, even if the agent didn't actually take it. That's what makes it **off-policy**.

### 4.2 CliffWalking — The Classic Test Environment

**CliffWalking** is the textbook environment to compare Q-Learning vs SARSA:

```
┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
│  │  │  │  │  │  │  │  │  │  │  │  │   Reward per step: -1
├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤   Fall off cliff: -100
│  │  │  │  │  │  │  │  │  │  │  │  │   Goal: reach G in bottom-right
├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤
│ S│CC│CC│CC│CC│CC│CC│CC│CC│CC│CC│ G│
└──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘
  S = Start    CC = Cliff (fall → -100, reset to S)    G = Goal
```

In [6]:
import gymnasium as gym
import numpy as np
import random

# ============================================================
# Q-Learning on CliffWalking-v1
# ============================================================

def q_learning(env_name, n_episodes=500, alpha=0.5, gamma=1.0,
               epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    """
    Q-Learning: Off-policy TD control.
    Learns the optimal Q-values regardless of the exploration policy.
    """
    env = gym.make(env_name)
    nS  = env.observation_space.n
    nA  = env.action_space.n
    
    Q       = np.zeros((nS, nA))
    epsilon = epsilon_start
    episode_rewards = []
    
    for ep in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            # ε-greedy action selection
            if random.random() < epsilon:
                action = env.action_space.sample()      # Explore
            else:
                action = int(np.argmax(Q[state]))       # Exploit
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            
            # ─── Q-LEARNING UPDATE ───────────────────────────────────────
            # Target: r + γ * max_a' Q(s', a')   ← Uses MAX over next actions
            td_target = reward + gamma * np.max(Q[next_state]) * (not done)
            td_error  = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            # ─────────────────────────────────────────────────────────────
            
            state = next_state
        
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)
    
    env.close()
    return Q, episode_rewards

print("=" * 55)
print("Q-LEARNING on CliffWalking-v1")
print("=" * 55)
print("Training for 500 episodes...")

Q_cliff, ql_rewards = q_learning('CliffWalking-v1', n_episodes=500)

# Print learning progress
print(f"\n  {'Episodes':>12}  {'Avg Reward':>12}")
print("  " + "-" * 26)
for chunk in range(5):
    start = chunk * 100
    avg   = np.mean(ql_rewards[start:start+100])
    print(f"  {start+1:>4}-{start+100:<4}      {avg:>+10.1f}")

# Show learned greedy policy
print("\nLearned Policy (4 rows × 12 cols):")
a_symbols = {0: chr(8593), 1: chr(8595), 2: chr(8594), 3: chr(8592)}  # ↑ ↓ → ←
for r in range(4):
    row = "  "
    for c in range(12):
        s = r * 12 + c
        if r == 3 and 1 <= c <= 10:   row += " C"   # Cliff
        elif r == 3 and c == 11:      row += " G"   # Goal
        else:                         row += f" {a_symbols[int(np.argmax(Q_cliff[s]))]}"
    print(row)
print("\n  ↑↓→← = direction | C = Cliff | G = Goal")
print("  Q-Learning learns the OPTIMAL path (hugging the cliff edge)")


Q-LEARNING on CliffWalking-v1
Training for 500 episodes...

      Episodes    Avg Reward
  --------------------------
     1-100          -7761.4
   101-200           -292.2
   201-300           -130.9
   301-400            -68.4
   401-500            -54.9

Learned Policy (4 rows × 12 cols):
   ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ →
   ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ →
   ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ →
   ↑ C C C C C C C C C C G

  ↑↓→← = direction | C = Cliff | G = Goal
  Q-Learning learns the OPTIMAL path (hugging the cliff edge)


### 4.3 Q-Learning on Taxi-v3

Let's test Q-Learning on a more complex environment: **Taxi**.

```
┌──────────────────────────────────────┐
│  The Taxi Problem                    │
│                                      │
│  • 5×5 grid = 25 taxi positions      │
│  • 4 passenger pickup locations      │
│  • 4 dropoff locations               │
│  • 5 passenger states                │
│  = 500 total states                  │
│                                      │
│  Actions: North, South, East, West,  │
│           Pick Up, Drop Off         │
│                                      │
│  Reward:  -1 per step               │
│           +20 for correct dropoff   │
│           -10 for wrong pickup/drop │
└──────────────────────────────────────┘
```

In [7]:
import gymnasium as gym
import numpy as np
import random

# ============================================================
# Q-Learning on Taxi-v3
# ============================================================

def q_learning_taxi(n_episodes=10000, alpha=0.1, gamma=0.99,
                    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.9995):
    env = gym.make('Taxi-v3')
    nS, nA = env.observation_space.n, env.action_space.n
    Q = np.zeros((nS, nA))
    epsilon = epsilon_start
    all_rewards = []
    all_steps   = []
    
    for ep in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        steps = 0
        done = False
        
        while not done:
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            steps += 1
            
            # Q-Learning update
            td_target         = reward + gamma * np.max(Q[next_state]) * (not done)
            Q[state][action] += alpha * (td_target - Q[state][action])
            state = next_state
        
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        all_rewards.append(total_reward)
        all_steps.append(steps)
    
    env.close()
    return Q, all_rewards, all_steps

print("=" * 55)
print("Q-LEARNING on Taxi-v3")
print("=" * 55)
print("Training for 10,000 episodes...\n")

Q_taxi, taxi_rewards, taxi_steps = q_learning_taxi(n_episodes=10_000)

print(f"{'Episodes':>14}  {'Avg Reward':>12}  {'Avg Steps':>10}")
print("-" * 42)
breakpoints = [(1, 500), (501, 1000), (1001, 2000), (2001, 5000), (5001, 10000)]
for start, end in breakpoints:
    avg_r = np.mean(taxi_rewards[start-1:end])
    avg_s = np.mean(taxi_steps[start-1:end])
    print(f"{start:>6}-{end:<6}     {avg_r:>+10.1f}  {avg_s:>10.1f}")

# Watch one episode of the trained agent
env_watch = gym.make('Taxi-v3')
state, _ = env_watch.reset(seed=7)
total_r, steps_taken = 0, 0
done = False
while not done:
    action = int(np.argmax(Q_taxi[state]))
    state, reward, terminated, truncated, _ = env_watch.step(action)
    total_r += reward
    steps_taken += 1
    done = terminated or truncated
env_watch.close()

print(f"\nTrained agent demo: {steps_taken} steps | reward = {total_r}")
print("(A random agent would take 200+ steps and score around -200)")
print("Q-Learning solved Taxi in ~13 steps on average after training!")

Q-LEARNING on Taxi-v3
Training for 10,000 episodes...

      Episodes    Avg Reward   Avg Steps
------------------------------------------
     1-500            -640.6       179.5
   501-1000           -266.8        94.3
  1001-2000            -56.9        32.5
  2001-5000             -4.5        16.7
  5001-10000            +6.4        13.5

Trained agent demo: 11 steps | reward = 10
(A random agent would take 200+ steps and score around -200)
Q-Learning solved Taxi in ~13 steps on average after training!


## 5. SARSA — On-Policy Temporal Difference Control

SARSA is Q-Learning's close cousin — with one critical difference:

| | Q-Learning | SARSA |
|---|---|---|
| **Type** | Off-policy | On-policy |
| **Update target** | `r + γ · max_a' Q(s', a')` | `r + γ · Q(s', a')` — actual *next action taken* |
| **Behavior** | Optimistic — assumes you'll take the best action | Conservative — accounts for your actual exploration |

### SARSA stands for: **(S)tate — (A)ction — (R)eward — (S)tate — (A)ction**

The name describes the tuple it needs for each update: $(s_t, a_t, r_{t+1}, s_{t+1}, a_{t+1})$

**The SARSA update rule**:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ r_{t+1} + \gamma \cdot Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t) \right]$$

Notice: the target uses $Q(s_{t+1}, a_{t+1})$ — the Q-value of the **action you actually took next** — not the max over all actions like Q-Learning.

### 5.1 The Cliff Walking Showdown

CliffWalking is the **classic example** that reveals the difference between Q-Learning and SARSA:

```
Q-Learning learns:       Safe path (but optimal — shorter route along the cliff edge)
                         ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑↗
                         →  →  →  →  →  →  →  →  →  →  → G
                         S  CC CC CC CC CC CC CC CC CC CC 

SARSA learns:            Safer path (further from the cliff — avoids accidental falls)
                         →  →  →  →  →  →  →  →  →  →  →↘
                         ↑  (wide berth from cliff)      ↓
                         S  CC CC CC CC CC CC CC CC CC CC G
```

Why? Because SARSA knows it will sometimes take random exploratory actions (due to ε-greedy). The cliff edge is dangerous when you're exploring — SARSA learns to stay away from it. Q-Learning ignores this and learns the optimal path assuming perfect behavior.

In [8]:
import gymnasium as gym
import numpy as np
import random

# ============================================================
# SARSA on CliffWalking-v1
# ============================================================

def sarsa(env_name, n_episodes=500, alpha=0.5, gamma=1.0,
          epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    """
    SARSA: On-policy TD control.
    Difference from Q-Learning: the update uses the ACTUAL next action taken.
    """
    env = gym.make(env_name)
    nS  = env.observation_space.n
    nA  = env.action_space.n
    Q   = np.zeros((nS, nA))
    epsilon = epsilon_start
    episode_rewards = []

    for ep in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        # ─── SARSA needs the action BEFORE the loop starts ───────────
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))
        # ─────────────────────────────────────────────────────────────

        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            # Choose next action (this is what makes it on-policy)
            if random.random() < epsilon:
                next_action = env.action_space.sample()
            else:
                next_action = int(np.argmax(Q[next_state]))

            # ─── SARSA UPDATE ────────────────────────────────────────────
            # Target: r + γ * Q(s', a')   ← Uses ACTUAL next action (not max)
            td_target = reward + gamma * Q[next_state][next_action] * (not done)
            td_error  = td_target - Q[state][action]
            Q[state][action] += alpha * td_error
            # ─────────────────────────────────────────────────────────────

            state  = next_state
            action = next_action   # Carry the action forward!

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)

    env.close()
    return Q, episode_rewards

print("=" * 55)
print("SARSA on CliffWalking-v1")
print("=" * 55)
print("Training for 500 episodes...")

Q_sarsa, sarsa_rewards = sarsa('CliffWalking-v1', n_episodes=500)

print(f"\n  {'Episodes':>12}  {'Avg Reward':>12}")
print("  " + "-" * 26)
for chunk in range(5):
    start = chunk * 100
    avg   = np.mean(sarsa_rewards[start:start+100])
    print(f"  {start+1:>4}-{start+100:<4}      {avg:>+10.1f}")

print("\nLearned Policy (SARSA):")
a_symbols = {0: chr(8593), 1: chr(8595), 2: chr(8594), 3: chr(8592)}  # ↑ ↓ → ←
for r in range(4):
    row = "  "
    for c in range(12):
        s = r * 12 + c
        if r == 3 and 1 <= c <= 10: row += " C"
        elif r == 3 and c == 11:    row += " G"
        else:                       row += f" {a_symbols[int(np.argmax(Q_sarsa[s]))]}"
    print(row)

print("\n  Notice: SARSA tends to take a SAFER path further from the cliff!")
print("  Compare with Q-Learning which prefers the riskier optimal path.")


SARSA on CliffWalking-v1
Training for 500 episodes...

      Episodes    Avg Reward
  --------------------------
     1-100          -8151.2
   101-200           -136.2
   201-300            -80.2
   301-400            -60.1
   401-500            -65.3

Learned Policy (SARSA):
   ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ →
   ↓ ↓ ↑ ↑ ↑ ↑ ↓ ↑ ↑ ↑ ↓ →
   ↑ ↑ ↑ ↑ ↑ ↑ ↑ ↑ ↑ ↑ ↓ →
   ↑ C C C C C C C C C C G

  Notice: SARSA tends to take a SAFER path further from the cliff!
  Compare with Q-Learning which prefers the riskier optimal path.


## 6. Comparing All Four Algorithms

Now let's put everything together and run a head-to-head comparison of all four algorithms on the same environment.

### 6.1 Q-Learning vs. SARSA on CliffWalking — Side by Side

This is the most famous comparison in RL textbooks (Sutton & Barto, Chapter 6).

In [9]:
import gymnasium as gym
import numpy as np
import random

# ============================================================
# Q-Learning vs SARSA — CliffWalking Performance Comparison
# ============================================================

# Re-run both with same settings for fair comparison
np.random.seed(42)
random.seed(42)

print("=" * 65)
print("Q-LEARNING vs SARSA — CliffWalking Performance")
print("=" * 65)
print()

N_EPISODES = 500
N_RUNS     = 10  # average over multiple runs to reduce noise

ql_all, sarsa_all = [], []

for run in range(N_RUNS):
    _, ql_r    = q_learning('CliffWalking-v1', n_episodes=N_EPISODES,
                             alpha=0.5, gamma=1.0, epsilon_start=0.1,
                             epsilon_end=0.1, epsilon_decay=1.0)
    _, sarsa_r = sarsa('CliffWalking-v1', n_episodes=N_EPISODES,
                        alpha=0.5, gamma=1.0, epsilon_start=0.1,
                        epsilon_end=0.1, epsilon_decay=1.0)
    ql_all.append(ql_r)
    sarsa_all.append(sarsa_r)

ql_mean    = np.mean(ql_all,    axis=0)
sarsa_mean = np.mean(sarsa_all, axis=0)

# Print comparison table in 50-episode chunks
print(f"{'Episode Range':>16}  {'Q-Learning Avg':>16}  {'SARSA Avg':>12}  {'Winner':>8}")
print("-" * 60)
for i in range(10):
    s, e = i * 50, (i + 1) * 50
    ql_avg    = np.mean(ql_mean[s:e])
    sarsa_avg = np.mean(sarsa_mean[s:e])
    winner    = "Q-L" if ql_avg > sarsa_avg else "SARSA"
    print(f"  {s+1:>4} - {e:<4}       {ql_avg:>+12.1f}    {sarsa_avg:>+10.1f}   {winner:>8}")

print()
print("INTERPRETATION:")
print("  • During training (with ε=0.1 exploration):")
print("    SARSA often achieves HIGHER per-episode rewards because")
print("    it accounts for its own exploration (stays safe from cliff)")
print()
print("  • At test time (ε=0, fully greedy):")
print("    Q-Learning would win because it found the shorter cliff-edge path")
print()
print("  → Q-Learning: optimistic | SARSA: realistic")


Q-LEARNING vs SARSA — CliffWalking Performance

   Episode Range    Q-Learning Avg     SARSA Avg    Winner
------------------------------------------------------------
     1 - 50               -115.3        -111.0      SARSA
    51 - 100               -49.1         -27.5      SARSA
   101 - 150               -50.6         -29.1      SARSA
   151 - 200               -51.9         -34.1      SARSA
   201 - 250               -50.3         -29.5      SARSA
   251 - 300               -55.3         -29.4      SARSA
   301 - 350               -50.0         -24.1      SARSA
   351 - 400               -46.2         -26.4      SARSA
   401 - 450               -53.7         -24.7      SARSA
   451 - 500               -50.2         -26.7      SARSA

INTERPRETATION:
  • During training (with ε=0.1 exploration):
    SARSA often achieves HIGHER per-episode rewards because
    it accounts for its own exploration (stays safe from cliff)

  • At test time (ε=0, fully greedy):
    Q-Learning would win b

### 6.2 All Four Algorithms on FrozenLake

Let's benchmark all four approaches on FrozenLake and compare their final win rates.

In [10]:
import gymnasium as gym
import numpy as np
import random
from collections import defaultdict

# ============================================================
# All 4 Algorithms — FrozenLake-v1 Benchmark
# ============================================================

def evaluate_q_table(Q, env_name, n_eval=1000, seed=0):
    """Evaluate a learned Q-table greedily (no exploration)."""
    env = gym.make(env_name)
    wins = 0
    for ep in range(n_eval):
        state, _ = env.reset(seed=seed + ep)
        done = False
        for _ in range(200):
            action = int(np.argmax(Q[state]))
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            if done:
                if reward == 1.0:
                    wins += 1
                break
    env.close()
    return wins / n_eval * 100

# ── 1. Dynamic Programming ──────────────────────────────────
env_dp = gym.make('FrozenLake-v1', is_slippery=True)
P_fl = env_dp.unwrapped.P
V_dp, dp_pol = dp_policy_iteration(P_fl, env_dp.observation_space.n,
                                    env_dp.action_space.n, gamma=0.99)
env_dp.close()
Q_dp = np.zeros((16, 4))
for s in range(16):
    Q_dp[s][dp_pol[s]] = 1.0  # Encode DP policy as Q-table
win_dp = evaluate_q_table(Q_dp, 'FrozenLake-v1')

# ── 2. Monte Carlo ──────────────────────────────────────────
env_mc = gym.make('FrozenLake-v1', is_slippery=True)
Q_mc   = defaultdict(lambda: np.zeros(4))
ret_mc = defaultdict(list)
eps_mc = 1.0
for ep in range(20000):
    episode = []
    obs, _ = env_mc.reset()
    done = False
    while not done:
        a = env_mc.action_space.sample() if random.random() < eps_mc else int(np.argmax(Q_mc[obs]))
        no, r, te, tr, _ = env_mc.step(a)
        episode.append((obs, a, r))
        done = te or tr
        obs = no
    G, vis = 0, set()
    for s_t, a_t, r_t in reversed(episode):
        G = r_t + 0.99 * G
        if (s_t, a_t) not in vis:
            vis.add((s_t, a_t))
            ret_mc[(s_t, a_t)].append(G)
            Q_mc[s_t][a_t] = np.mean(ret_mc[(s_t, a_t)])
    eps_mc = max(0.01, eps_mc * 0.9999)
env_mc.close()
Q_mc_arr = np.array([Q_mc[s] for s in range(16)])
win_mc = evaluate_q_table(Q_mc_arr, 'FrozenLake-v1')

# ── 3. Q-Learning ───────────────────────────────────────────
Q_ql_fl, _ = q_learning('FrozenLake-v1', n_episodes=10000,
                          alpha=0.8, gamma=0.99,
                          epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.9995)
win_ql = evaluate_q_table(Q_ql_fl, 'FrozenLake-v1')

# ── 4. SARSA ────────────────────────────────────────────────
Q_sarsa_fl, _ = sarsa('FrozenLake-v1', n_episodes=10000,
                       alpha=0.8, gamma=0.99,
                       epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.9995)
win_sarsa = evaluate_q_table(Q_sarsa_fl, 'FrozenLake-v1')

# ── Print Results ───────────────────────────────────────────
print("=" * 65)
print("ALL 4 ALGORITHMS ON FrozenLake-v1 (1,000 evaluation episodes)")
print("=" * 65)
print(f"\n  {'Algorithm':<25} {'Win Rate':>10}  {'Notes'}")
print("  " + "-" * 60)
results = [
    ("Dynamic Programming",     win_dp,    "Needs model; exact optimal policy"),
    ("Monte Carlo",             win_mc,    "No model; needs complete episodes"),
    ("Q-Learning (off-policy)", win_ql,    "No model; updates every step"),
    ("SARSA (on-policy)",       win_sarsa, "No model; conservative updates"),
]
for name, rate, note in results:
    print(f"  {name:<25} {rate:>8.1f}%  {note}")

best = max(results, key=lambda x: x[1])
print(f"\n  Best performer: {best[0]} ({best[1]:.1f}%)")

  Policy Iteration converged in 7 iterations.
ALL 4 ALGORITHMS ON FrozenLake-v1 (1,000 evaluation episodes)

  Algorithm                   Win Rate  Notes
  ------------------------------------------------------------
  Dynamic Programming           75.5%  Needs model; exact optimal policy
  Monte Carlo                   59.8%  No model; needs complete episodes
  Q-Learning (off-policy)       20.1%  No model; updates every step
  SARSA (on-policy)             46.3%  No model; conservative updates

  Best performer: Dynamic Programming (75.5%)


### 6.3 Algorithm Decision Guide

Use this table to choose the right algorithm for your problem:

| Situation | Best Choice | Why |
|---|---|---|
| You have the **full model** (P, R known) | **Dynamic Programming** | Exact optimal solution; no sampling needed |
| **No model**, episodes are **short** and well-defined | **Monte Carlo** | Simple to implement; uses full unbiased returns |
| **No model**, **continuous** or **long** episodes | **Q-Learning or SARSA** | Updates every step; doesn't need to wait |
| Safety is critical; agent must **not make dangerous mistakes** | **SARSA** | On-policy: accounts for actual exploration behavior |
| You want the **fastest convergence** to optimal | **Q-Learning** | Off-policy: directly targets optimal Q-values |
| Large state space needing generalization | **DQN** (Part 4) | Uses a neural network to approximate Q-values |

### Complexity Comparison

```
           Needs     Episode  Bootstrap   On-Policy  Convergence
Algorithm   Model?   Based?   (not MC)?   Update?     Guarantee
──────────  ──────   ───────  ─────────   ─────────   ──────────
DP           YES       No       YES          N/A        YES (exact)
Monte Carlo  No        YES      No          YES         YES (GLIE)
Q-Learning   No        No       YES          No         YES (tabular)
SARSA        No        No       YES         YES         YES (tabular)
```

## 7. ✏️ Exercise: Implement Q-Learning on a New Environment

Now it's your turn! Implement Q-Learning on **CliffWalking** with a twist: experiment with different hyperparameters and compare the results.

### Your Tasks

1. Train Q-Learning with `alpha=0.1` and `alpha=0.9` — which learns faster?
2. Try `gamma=0.9` vs `gamma=1.0` — which finds a shorter path?
3. Compare the final learned policies visually

Modify the `TODO` sections in the cell below:

In [11]:
import gymnasium as gym
import numpy as np
import random

# ============================================================
# Exercise: Experiment with Q-Learning Hyperparameters
# ============================================================
# Modify the parameters below and observe how they affect learning.

experiments = [
    # (label, alpha, gamma, epsilon)
    ("Low LR   (α=0.1, γ=1.0)",  0.1, 1.0, 0.1),   # TODO: Try changing these
    ("High LR  (α=0.9, γ=1.0)",  0.9, 1.0, 0.1),
    ("Discount (α=0.5, γ=0.9)",  0.5, 0.9, 0.1),
    ("Greedy   (α=0.5, γ=1.0, ε=0.01)", 0.5, 1.0, 0.01),
]

print("=" * 65)
print("EXERCISE: Hyperparameter Comparison on CliffWalking")
print("=" * 65)
print()

a_symbols = {0: chr(8593), 1: chr(8595), 2: chr(8594), 3: chr(8592)}

for label, alpha, gamma, eps in experiments:
    Q_exp, rewards_exp = q_learning(
        'CliffWalking-v1',
        n_episodes=500,
        alpha=alpha,
        gamma=gamma,
        epsilon_start=eps,
        epsilon_end=eps,
        epsilon_decay=1.0       # constant epsilon
    )
    avg_last_100 = np.mean(rewards_exp[-100:])
    
    print(f"  Config: {label}")
    print(f"  Avg reward (last 100 episodes): {avg_last_100:+.1f}")
    
    # Print policy grid for top 2 rows (summary)
    for r in range(2):
        row = "    "
        for c in range(12):
            s = r * 12 + c
            row += f" {a_symbols[int(np.argmax(Q_exp[s]))]}"
        print(row)
    print()

print("Which config found the best path? Explain in your own words!")
print()
print("DISCUSSION QUESTIONS:")
print("  1. Why does a high alpha (0.9) sometimes learn faster but converge poorly?")
print("  2. Why does gamma=0.9 produce a different path than gamma=1.0 in CliffWalking?")
print("  3. What happens with epsilon=0.01 from the start? Why is exploration important?")


EXERCISE: Hyperparameter Comparison on CliffWalking

  Config: Low LR   (α=0.1, γ=1.0)
  Avg reward (last 100 episodes): -49.1
     ↓ ↑ ↓ ↑ ↓ ↓ ↓ ↓ → ↓ → →
     ↓ ↑ ↓ ↓ ↓ → ↓ ↓ → → ↓ →

  Config: High LR  (α=0.9, γ=1.0)
  Avg reward (last 100 episodes): -52.6
     ← ↓ ↓ ← ↓ ↓ ↓ ↓ → ↓ ↓ →
     ↓ ↓ ↓ → ↓ → ↓ ↓ ↓ ↓ ↓ →

  Config: Discount (α=0.5, γ=0.9)
  Avg reward (last 100 episodes): -47.1
     ↑ ↓ ↓ ↓ → ↓ → ↓ ↓ ↓ → →
     ← → → ↓ ↓ ↓ ↓ → → ↓ → →

  Config: Greedy   (α=0.5, γ=1.0, ε=0.01)
  Avg reward (last 100 episodes): -18.4
     ↑ ↓ ↑ ← ↑ ↓ → ← ↓ ↑ ↓ →
     ↓ ↓ ↓ ↓ ↓ ↓ → ↓ ↓ ↓ ↓ →

Which config found the best path? Explain in your own words!

DISCUSSION QUESTIONS:
  1. Why does a high alpha (0.9) sometimes learn faster but converge poorly?
  2. Why does gamma=0.9 produce a different path than gamma=1.0 in CliffWalking?
  3. What happens with epsilon=0.01 from the start? Why is exploration important?


## 8. Summary & Key Takeaways

### What We Learned Today

✅ **Dynamic Programming** — perfect when you have the model; Policy Iteration alternates between evaluating and improving a policy until it converges  

✅ **Monte Carlo** — model-free; learns from complete episodes by averaging returns; First-Visit MC is simple and effective  

✅ **Q-Learning** — model-free, off-policy TD; updates at every step; targets the greedy `max` over next actions; finds the optimal policy efficiently  

✅ **SARSA** — model-free, on-policy TD; uses the actual next action taken; more conservative and safer than Q-Learning near dangerous states  

✅ **When to use what** — DP for known models, MC for episodic tasks, Q-Learning for optimal convergence, SARSA for safe exploration  

---

### Key Terms Cheat Sheet

| Term | Plain English |
|---|---|
| **Dynamic Programming** | Solve MDPs exactly when you know all probabilities and rewards |
| **Policy Iteration** | Alternate between evaluating a policy and improving it |
| **Monte Carlo** | Learn from complete episode outcomes — no model, no bootstrapping |
| **Return G** | Total discounted reward from a given step to the end of an episode |
| **Temporal Difference (TD)** | Update estimates at every step using the next state's value |
| **Bootstrapping** | Using your own current estimates as part of the update target |
| **Q-Learning** | Off-policy TD: targets the best possible next action (max Q) |
| **SARSA** | On-policy TD: targets the action you actually took next |
| **Off-policy** | Learns about the optimal policy while following a different (exploratory) policy |
| **On-policy** | Learns about the same policy it's currently using to make decisions |
| **ε-greedy** | Take the best known action with prob (1−ε); explore randomly with prob ε |

---

### Coming Up in Part 4: Deep RL (DQN, Policy Gradient, PPO)

All four algorithms we learned today use **Q-tables** — they store one value per (state, action) pair. This works perfectly for small environments (16 states, 4 actions).

But what about environments with millions or infinite states? Like Atari games with raw pixel inputs (3.2 million possible states per frame)?

The answer: **replace the Q-table with a neural network**! In Part 4, we'll learn:
1. **DQN** — Deep Q-Networks; Q-Learning with a neural network as Q-function approximator
2. **Experience Replay** — why you can't just train on consecutive steps
3. **Target Networks** — how to stabilize deep RL training
4. **Policy Gradient Methods** — a completely different approach: learn the policy directly!
5. **PPO** — Proximal Policy Optimization; the algorithm that powers ChatGPT's RLHF tuning

---

*© 2025 Reinforcement Learning for AI/ML Engineers — Mehdi Lotfinejad*

## 📋 Quick Reference Card

```
┌─────────────────────────────────────────────────────────────────────┐
│             KEY RL ALGORITHMS CHEAT SHEET                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  DYNAMIC PROGRAMMING (model required)                               │
│  Policy Iteration:  Evaluate π → Improve π greedily → Repeat       │
│  Value Iteration:   V(s) ← max_a [R + γ·V(s')]  until stable        │
│                                                                     │
│  MONTE CARLO (no model, episodic)                                   │
│  1. Play full episode                                               │
│  2. Compute return G backwards:  G_t = r_{t+1} + γ·G_{t+1}         │
│  3. Update Q(s,a) ← average of all observed G values               │
│  4. Improve policy greedily                                         │
│                                                                     │
│  Q-LEARNING (off-policy TD)                                         │
│  Q(s,a) ← Q(s,a) + α [r + γ·max_a' Q(s',a') - Q(s,a)]             │
│  Key: uses MAX over all next actions (off-policy!)                  │
│                                                                     │
│  SARSA (on-policy TD)                                               │
│  Q(s,a) ← Q(s,a) + α [r + γ·Q(s',a') - Q(s,a)]                    │
│  Key: uses ACTUAL next action taken (on-policy!)                    │
│                                                                     │
│  HYPERPARAMETERS:                                                   │
│  α (alpha) = learning rate  — how fast to update (0.1 to 0.9)      │
│  γ (gamma) = discount factor — how patient  (0.9 to 0.99)          │
│  ε (epsilon)= exploration   — how random  (start 1.0, decay to 0)  │
│                                                                     │
│  DECISION GUIDE:                                                    │
│  Know the model?         → Dynamic Programming                      │
│  Episodic, no model?     → Monte Carlo                              │
│  Step-by-step, optimal?  → Q-Learning                               │
│  Step-by-step, safe?     → SARSA                                    │
│  Large state space?      → DQN (Part 4!)                            │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```